> ## SOLUTION / ANSWER KEY &mdash; Lab 7.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-email-drafting-agent.ipynb`](../lab-12-capstone-email-drafting-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.12 &mdash; Capstone: The Email-Drafting Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Chain the whole pipeline: extract, route, gather, draft, validate
- Never auto-send -- every result is a needs_approval draft
- Run it over a SUITE of client emails and score the outcomes

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Now build it — Module 7 labs](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-12"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Capstone: the **email-drafting agent** (the client's Lab 4.1), end to end. It **extracts** the
query's fields, **routes** it to a team, **gathers** the order + template, **drafts** a grounded
reply, **validates** it, and returns a **`needs_approval`** draft &mdash; it **never auto-sends**. You
run it over a **suite** of incoming emails and score the outcomes. The helpers below are the ones you
built through the module; you assemble them into `process` and score with `evaluate`.

In [ ]:
class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

# The pieces you built this module, provided here so you can assemble the whole agent.
def extract(email):
    text = email.lower()
    digits = "".join(ch for ch in email if ch.isdigit())
    order_id = digits if digits else None
    if "refund" in text: intent = "refund"
    elif any(w in text for w in ("deliver", "arrive", "late", "where is")): intent = "delivery"
    elif "cancel" in text: intent = "cancel"
    else: intent = "other"
    sentiment = "negative" if any(w in text for w in ("frustrated", "angry", "late", "still")) else "neutral"
    return {"order_id": order_id, "intent": intent, "sentiment": sentiment}

TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
def route(rec):
    return {"team": TEAMS.get(rec["intent"], "general"),
            "escalate": rec["sentiment"] == "negative" or rec["intent"] not in TEAMS}

TEMPLATE_FOR = {"delivery": "delivery_delay", "refund": "refund"}
def draft(order, intent):
    kind = TEMPLATE_FOR.get(intent, "delivery_delay")
    return PromptTemplate.from_template(TEMPLATES[kind]).format(
        name=order["name"], id=order["id"], status=order["status"], eta=order["eta"])
def validate(reply, order):
    return order["eta"] in reply
print("helpers ready: extract, route, draft, validate + ORDERS/TEMPLATES")

## Your Turn
Assemble `process` (chain the pipeline; never send) and `evaluate` (score the suite).

In [ ]:
def process(email):
    rec    = extract(email)
    routed = route(rec)
    order  = ORDERS.get(rec["order_id"], {"id": rec["order_id"], "name": "there", "status": "unknown", "eta": "soon"})
    reply  = draft(order, rec["intent"])
    ok     = validate(reply, order)
    # never auto-send: a valid draft awaits approval; an invalid one needs a fix
    status = "needs_approval" if ok else "needs_fix"
    return {"team": routed["team"], "escalate": routed["escalate"],
            "draft": reply, "status": status}

SUITE = [
    {"email": "Where is my order 4471? It's late.", "team": "logistics"},
    {"email": "Please refund order 5090",           "team": "billing"},
]

def evaluate():
    # count suite emails routed to the expected team AND left as a draft (never sent)
    solved = sum(1 for t in SUITE if process(t["email"])["team"] == t["team"] and process(t["email"])["status"].startswith("needs_"))
    return solved, len(SUITE)

try:
    for t in SUITE:
        r = process(t['email'])
        print(t['email'][:28], '->', r['team'], '|', r['status'])
    print('score:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a delivery email routes to logistics", lambda: process("Where is my order 4471? It's late.")["team"] == "logistics")
expect_true("a refund email routes to billing", lambda: process("Please refund order 5090")["team"] == "billing")
expect_true("the delivery draft is grounded (ETA present)", lambda: "Friday" in process("Where is my order 4471? It's late.")["draft"])
expect_true("NO email is ever auto-sent (always needs_*)", lambda: all(process(t["email"])["status"].startswith("needs_") for t in SUITE))
expect_true("a frustrated customer is escalated", lambda: process("I am frustrated, order 4471 still late")["escalate"] is True)
expect_true("evaluate solves the whole suite", lambda: evaluate() == (2, 2))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
Swap the scripted pieces for a REAL LangChain email agent (Ollama / Groq) -- the bridge to Module 8. Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`. If a package, model or key is
missing the cell prints a friendly note and moves on.
**The graded steps above are offline &amp; deterministic, so the lab always verifies without a model.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    email = "Where is my order 4471? It's late."
    reply = llm.invoke("You are a support agent. Draft a short, polite reply (do NOT send): " + email).content
    print("REAL drafted reply:\n", reply)
    print("\nProduction shape: extract -> route -> gather (tools) -> draft (llm) -> validate -> human approves -> send.")
    print("Next: Module 8 -- when one agent isn't enough, route to specialist AGENTS (the customer-service chatbot).")
except Exception as e:
    print("No local LLM available -- skipping (pip install langchain langchain-ollama + `ollama run llama3.2:1b`,")
    print("or langchain-groq with GROQ_API_KEY):", type(e).__name__)
    print("The offline email-drafting agent above already ran the whole suite -- extract, route, draft, validate,")
    print("and returned needs_approval drafts (never auto-sent). Next: Module 8 -- multi-agent collaboration.")

---
### Key takeaway
You built the email-drafting agent end to end -- extract, route, gather, draft, validate -- and it never sends on its own. That's an agent that does a job you'd pay for. Next: Module 8 orchestrates a team of them.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>